# Week 2: Advanced Pandas Data Wrangling
## Notebook 10: MultiIndex & Hierarchical DataFrames

### 1. Conceptual Intuition & Theoretical Background
Real-world datasets frequently encompass nested hierarchies (e.g., Country $\to$ State $\to$ City, or Year $\to$ Quarter $\to$ Month).

A **MultiIndex (Hierarchical Index)** represents higher-dimensional tensors within 2D tabular DataFrames:
- Tuple-based level addressing: $(\text{Level}_0, \text{Level}_1, \dots, \text{Level}_k)$.
- **Lexicographical Sorting:** Operations like partial slicing require indices to be sorted via `.sort_index()` for $O(\log N)$ lookup performance.
- **Cross-Sectional Slicing (`.xs`):** Extracts sub-slices across any internal index level without resetting index structure.

Transformation dualities:
$$\text{Unstack: Row Levels } \longrightarrow \text{ Column Levels}$$
$$\text{Stack: Column Levels } \longrightarrow \text{ Row Levels}$$

---

### 2. Scratch Python Implementation: MultiIndex Data Engine

In [ ]:
import pandas as pd
import numpy as np

# Construct MultiIndex Tuples: (Region, Store, Department)
regions = ['North', 'South']
stores = ['Store_A', 'Store_B']
departments = ['Electronics', 'Clothing']

index_tuples = [
    (r, s, d)
    for r in regions
    for s in stores
    for d in departments
]

multi_idx = pd.MultiIndex.from_tuples(index_tuples, names=['Region', 'Store', 'Department'])

np.random.seed(42)
df = pd.DataFrame({
    'Revenue': np.random.uniform(20000, 80000, size=len(index_tuples)).round(2),
    'Customer_Count': np.random.randint(100, 500, size=len(index_tuples))
}, index=multi_idx).sort_index()

# 1. Cross-Sectional Retrieval (.xs) across internal level 'Department'
clothing_slice = df.xs('Clothing', level='Department')

# 2. Level-based Aggregation: Mean Revenue by Region and Department
regional_dept_summary = df.groupby(level=['Region', 'Department'])['Revenue'].mean()

# 3. Unstacking: Reshape internal index level into pivot columns
unstacked_view = df['Revenue'].unstack(level='Department')

print("--- Hierarchical MultiIndex DataFrame ---")
print(df.head(4))
print("\n--- Cross-Section (.xs) for 'Clothing' Department ---")
print(clothing_slice)
print("\n--- Unstacked Pivot View (Departments as Columns) ---")
print(unstacked_view)
print(f"\nIndex Lexicographical Sorted Check: {df.index.is_monotonic_increasing}")

### 3. Step-by-Step Code Walkthrough
1. **`from_tuples` Construction:** Creates a 3-level hierarchical index structure.
2. **`.xs()` Selection:** Retrieves records where `Department == 'Clothing'` without needing Boolean filtering masks.
3. **`.unstack()`:** Pivots a row index level into horizontal column headers.

---

### 4. Industry Applications & Trade-offs
- **Financial Tick & Portfolio Analytics:** Indexing securities by `[Sector, Ticker, Timestamp]`.
- **Trade-off:** Unsorted MultiIndexes cause performance degradation and throw `UnsortedIndexError` on range slices.

---

### 5. Top Interview Questions & Answers
**Q1: Why is `df.sort_index()` mandatory before performing complex MultiIndex range slicing?**
> **Answer:** Pandas uses binary search on sorted indices ($O(\log N)$). On unsorted indices, slicing requires expensive full-table scans or fails outright.

**Q2: What is the difference between `df.unstack()` and `df.pivot_table()`?**
> **Answer:** `.unstack()` is a fast reshape operation operating strictly on existing index levels without aggregation. `.pivot_table()` is a generalized aggregation tool that can aggregate duplicate keys.